# Zero-shot top-down vs bottom-up exploration

This notebook loads the ISCO multilingual test sentences, derives the inference
text for each sample, and calls the same Kedro node functions used in production
pipelines to compute the top-down routing and bottom-up validation routes.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import yaml

from taxomind.pipelines.zero_shot import nodes as zero_nodes

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / "data"
CONF_DIR = PROJECT_DIR / "conf"

print(f"Project directory: {PROJECT_DIR}")


Project directory: /Users/gabriele/App/rowsquared/taxomind


In [2]:
test_path = DATA_DIR / "01_raw" / "isco_test_sentences.json"
taxonomy_path = DATA_DIR / "05_model_input" / "taxonomy_embedded.parquet"
parameters_path = CONF_DIR / "base" / "parameters.yml"

with test_path.open(encoding="utf-8") as fp:
    test_payload = json.load(fp)
taxonomy_df = pd.read_parquet(taxonomy_path)
with parameters_path.open(encoding="utf-8") as fp:
    parameters = yaml.safe_load(fp)

model_name = parameters["zero_shot"]["model_name"]
print(f"Loaded {len(test_payload.get('sentences', []))} test sentences.")
print(f"Taxonomy rows: {len(taxonomy_df):,}")
print(f"Embedding model: {model_name}")


Loaded 58 test sentences.
Taxonomy rows: 619
Embedding model: BAAI/bge-m3


In [4]:
sentences = []
for record in test_payload.get("sentences", []):
    inference_text = zero_nodes.compose_inference_text(record.get("fields", {}))
    sentences.append(
        {
            "sentence_id": record.get("sentence_id"),
            "taxonomyKey": test_payload.get("taxonomyKey"),
            "text": inference_text,
        }
    )

sentences_df = pd.DataFrame(sentences)
sentences_df

,sentence_id,taxonomyKey,text
0,f61a4ea1-6c9b-4cee-b307-9ee236aacc0b,ISCO,Industry Description: agriculture own business...
1,10633c10-75a0-4269-9788-454bd4365507,ISCO,Industry Description: Ministry of Education\nJ...
2,afe88398-5435-4954-a216-9d6293ae1a9f,ISCO,Industry Description: Construction\nJob Descri...
3,d92f3cef-4029-43bc-889e-117a169ef7a5,ISCO,Industry Description: linus farm\nJob Descript...
4,2542a3a8-03b6-49c1-a6dc-8b7793e89a67,ISCO,Industry Description: wind jammer\nJob Descrip...
5,c2861742-7216-48e9-8407-148909a20abb,ISCO,Industry Description: Fantastic Cuisine\nJob D...
6,be89e8d8-7e8b-40b8-a353-14e35a622aef,ISCO,Industry Description: cricket fields\nJob Desc...
7,37ddd657-fb72-4c38-b717-fbddd99c6bd4,ISCO,Industry Description: Automotive art\nJob Desc...
8,9b2dfd1c-820e-4fd3-bfed-3a42a096bace,ISCO,Industry Description: Antoine s farm\nJob Desc...
9,74c1e290-de3e-42af-8c96-f571552bf760,ISCO,Industry Description: Babonneau(planting of gr...


In [5]:
results = []
for row in sentences_df.itertuples(index=False):
    topdown = zero_nodes.top_down_route(row.text, taxonomy_df, model_name)
    bottomup = zero_nodes.bottom_up_validation(row.text, taxonomy_df, topdown, model_name)
    results.append(
        {
            "sentence_id": row.sentence_id,
            "text": row.text,
            "top_down_route": topdown.get("route", []),
            "bottom_up_route": bottomup.get("route", []),
            "routes_match": bottomup.get("routes_match"),
        }
    )

print(f"Computed routes for {len(results)} sentences.")


Computed routes for 58 sentences.


In [6]:
example = results[0]
print(f"Sentence ID: {example['sentence_id']}")
print(example["text"])
pd.DataFrame(example["top_down_route"])


Sentence ID: f61a4ea1-6c9b-4cee-b307-9ee236aacc0b
Industry Description: agriculture own business
Job Description: mixed vegetable farmer


,code,label,score,level,parentCode,isLeaf,language
0,6,"Skilled Agricultural, Forestry and Fishery Wor...",0.567194,1,None,False,None
1,61,Market-oriented Skilled Agricultural Workers,0.590220,2,6,False,None
2,613,Mixed Crop and Animal Producers,0.656536,3,61,False,None
3,6130,Mixed Crop and Animal Producers,0.685794,4,613,True,None


In [7]:
pd.DataFrame(example["bottom_up_route"])

,code,label,score,level,parentCode,isLeaf,language
0,6,"Skilled Agricultural, Forestry and Fishery Wor...",0.567194,1,None,False,None
1,61,Market-oriented Skilled Agricultural Workers,0.590220,2,6,False,None
2,611,Market Gardeners and Crop Growers,0.594048,3,61,False,None
3,6114,Mixed Crop Growers,0.707040,4,611,True,None
